# CIF суперячейки ХСИС Al-Co-Cr-Fe-Ni-Ti
Упрочняющая фаза **(Ni,Co,Fe,Cr)₃(Al,Ti)** со структурой **L1₂** (Pm-3m, #221)

### Структура L1₂ (тип Cu₃Au)
| Сайт | Позиция | Элементы |
|---|---|---|
| **1a** (0,0,0) | B-сайт | Al, Ti |
| **3c** (½,½,0), (½,0,½), (0,½,½) | A-сайт | Ni, Co, Fe, Cr |

### Для DFT: конкретная конфигурация
Дробные заселённости заменяются на **фиксированные позиции** атомов в суперячейке 2×2×2 (32 атома).
Атомы расставляются псевдослучайно с сохранением целевого состава.

In [ ]:
%pip install pymatgen

In [ ]:
from pymatgen.core import Structure, Lattice
from pymatgen.io.cif import CifWriter
import numpy as np
import random

## 1. Параметры

In [ ]:
# Параметр решётки L12 (Ni3Al ≈ 3.57 A)
a = 3.57

# Целевой состав A-сайта (Ni,Co,Fe,Cr) - доли должны суммироваться в 1
a_site_composition = {"Ni": 0.50, "Co": 0.20, "Fe": 0.15, "Cr": 0.15}

# Целевой состав B-сайта (Al,Ti)
b_site_composition = {"Al": 0.75, "Ti": 0.25}

# Размер суперячейки
n = 2  # 2x2x2 -> 32 атома (8 B-сайтов + 24 A-сайта)

## 2. Суперячейка + конкретная расстановка атомов

In [ ]:
def build_l12_supercell(a, n, a_comp, b_comp, seed=42):
    """
    Строит суперячейку L12 n x n x n с конкретной расстановкой атомов.
    
    L12 (Pm-3m): B-сайт на (0,0,0), A-сайты на (1/2,1/2,0), (1/2,0,1/2), (0,1/2,1/2)
    В суперячейке: каждая 4-я позиция (i%4==0) = B-сайт, остальные = A-сайт
    """
    random.seed(seed)
    np.random.seed(seed)

    lattice = Lattice.cubic(a * n)
    
    # Генерируем все позиции суперячейки
    all_species = []
    all_coords = []
    
    for ix in range(n):
        for iy in range(n):
            for iz in range(n):
                # B-сайт (1a)
                all_species.append("Al")  # placeholder
                all_coords.append([(ix + 0) / n, (iy + 0) / n, (iz + 0) / n])
                # A-сайты (3c)
                all_species.append("Ni")  # placeholder
                all_coords.append([(ix + 0.5) / n, (iy + 0.5) / n, (iz + 0) / n])
                all_species.append("Ni")  # placeholder
                all_coords.append([(ix + 0.5) / n, (iy + 0) / n, (iz + 0.5) / n])
                all_species.append("Ni")  # placeholder
                all_coords.append([(ix + 0) / n, (iy + 0.5) / n, (iz + 0.5) / n])
    
    supercell = Structure(lattice, all_species, all_coords)
    
    # Индексы B и A сайтов
    b_indices = [i for i in range(len(supercell)) if i % 4 == 0]
    a_indices = [i for i in range(len(supercell)) if i % 4 != 0]
    
    n_b = len(b_indices)
    n_a = len(a_indices)
    print(f"B-сайтов: {n_b}, A-сайтов: {n_a}")

    # Распределяем элементы по B-сайтам
    b_elements = []
    for el, frac in b_comp.items():
        count = round(frac * n_b)
        b_elements.extend([el] * count)
    while len(b_elements) < n_b:
        b_elements.append(list(b_comp.keys())[0])
    b_elements = b_elements[:n_b]
    random.shuffle(b_elements)

    # Распределяем элементы по A-сайтам
    a_elements = []
    for el, frac in a_comp.items():
        count = round(frac * n_a)
        a_elements.extend([el] * count)
    while len(a_elements) < n_a:
        a_elements.append(list(a_comp.keys())[0])
    a_elements = a_elements[:n_a]
    random.shuffle(a_elements)

    # Заменяем элементы
    for idx, el in zip(b_indices, b_elements):
        supercell.replace(idx, el)
    for idx, el in zip(a_indices, a_elements):
        supercell.replace(idx, el)

    return supercell


supercell = build_l12_supercell(
    a, n, a_site_composition, b_site_composition, seed=42
)

print(f"\nФормула: {supercell.composition.formula}")
print(f"Приведенная формула: {supercell.composition.reduced_formula}")
print(f"Число атомов: {supercell.num_sites}")
print(f"Параметры решетки: {supercell.lattice.a:.4f} A")
print(f"\nСостав по элементам:")
for el, amt in supercell.composition.as_dict().items():
    print(f"  {el}: {amt:.0f} atomov ({amt/supercell.num_sites*100:.1f}%)")

## 3. Экспорт в CIF (DFT-совместимый)

In [ ]:
writer = CifWriter(supercell, symprec=0.1)
writer.write_file("HEA_L12_supercell_2x2x2.cif")
print("Сохранен: HEA_L12_supercell_2x2x2.cif")

## 4. Проверка: позиции атомов

In [ ]:
print("Позиции атомов (дробные координаты):")
print(f"{'#':>3}  {'Element':>7}  {'x':>10}  {'y':>10}  {'z':>10}")
print("-" * 45)
for i, site in enumerate(supercell):
    fc = site.frac_coords
    print(f"{i+1:>3}  {site.species_string:>7}  {fc[0]:>10.4f}  {fc[1]:>10.4f}  {fc[2]:>10.4f}")